In [6]:
# If you don't have reportlab yet, uncomment the next line:
# !pip install reportlab

from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    ListFlowable,
    ListItem,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from datetime import datetime

# ---------------------------
# 1. Define curriculum outline
# ---------------------------

sessions = [
    {
        "title": "Session 1 — Environment, Tools, and Tips Dataset Overview",
        "goals": [
            "Set up Python, Jupyter, conda environment, and project folder structure.",
            "Load the 'tips' dataset into a pandas DataFrame.",
            "Understand each column and basic data types."
        ],
        "concepts": [
            "Using conda environments and requirements/environment.yml.",
            "Reading CSVs with pandas (read_csv).",
            "Inspecting data: head(), info(), describe().",
            "Basic data cleaning and renaming columns."
        ],
    },
    {
        "title": "Session 2 — EDA, Feature Engineering, and Simple Visuals",
        "goals": [
            "Create new features like tip percentage of bill.",
            "Explore distributions and relationships with simple plots.",
            "Get comfortable moving between code, charts, and interpretation."
        ],
        "concepts": [
            "Feature engineering (tip_pct = tip_usd / bill_total_usd).",
            "Histograms, bar plots, and boxplots (seaborn/matplotlib).",
            "Grouping and aggregation with groupby().",
            "Writing ELI5-style interpretations of plots."
        ],
    },
    {
        "title": "Session 3 — Correlation and Simple Linear Regression",
        "goals": [
            "Quantify linear relationships between numeric variables.",
            "Fit a simple linear regression: tip_usd ~ bill_total_usd.",
            "Interpret slope, intercept, and R-squared in plain language."
        ],
        "concepts": [
            "Pearson correlation vs. Spearman correlation.",
            "Scatter plots with trend lines (sns.lmplot / regplot).",
            "scikit-learn LinearRegression: fit, predict, coef_, intercept_.",
            "Meaning of R² and adjusted R²."
        ],
    },
    {
        "title": "Session 4 — Multiple Regression and Categorical Variables",
        "goals": [
            "Add additional predictors like party_size, gender, is_smoker.",
            "Encode categorical variables correctly for regression.",
            "Compare simple vs. multiple regression models."
        ],
        "concepts": [
            "Dummy variables (one-hot encoding) for gender and smoker.",
            "Reading regression coefficients while holding others constant.",
            "Standardized coefficients for feature importance.",
            "Train/test split for basic model validation."
        ],
    },
    {
        "title": "Session 5 — Residual Diagnostics and Assumptions",
        "goals": [
            "Check if linear regression assumptions are roughly satisfied.",
            "Use residual plots to look for patterns and problems.",
            "Understand normality, homoscedasticity, and endogeneity at a high level."
        ],
        "concepts": [
            "Residuals vs predicted plots (looking for random cloud around 0).",
            "Residual histograms and Q–Q plots for normality.",
            "Homoscedasticity vs heteroscedasticity.",
            "Endogeneity (feature correlated with error term) and why it biases OLS."
        ],
    },
    {
        "title": "Session 6 — Transformations and Robustness",
        "goals": [
            "Experiment with log-transforming the target (log(tip)).",
            "Compare linear vs log-transformed models using RMSE and R².",
            "See when a transformation helps or does not help."
        ],
        "concepts": [
            "Log and log1p transforms for skewed variables.",
            "Interpreting coefficients in log models in plain language.",
            "RMSE as a scale-dependent error metric.",
            "Trade-offs: better residual shape vs easier interpretability."
        ],
    },
    {
        "title": "Session 7 — Multicollinearity and VIF",
        "goals": [
            "Detect when predictors are telling the same story.",
            "Compute Variance Inflation Factor (VIF) for each feature.",
            "Decide when multicollinearity is a concern and what to do about it."
        ],
        "concepts": [
            "Correlation matrix heatmaps for features.",
            "Variance Inflation Factor (VIF) and typical thresholds.",
            "Impact of multicollinearity on coefficient stability.",
            "Strategies: dropping features, combining features, or using regularization (preview)."
        ],
    },
    {
        "title": "Session 8 — Model Comparison and Baselines",
        "goals": [
            "Define a simple baseline model to compare against.",
            "Compare multiple models using the same metrics.",
            "Summarize model performance in a small, clear table and narrative."
        ],
        "concepts": [
            "Baselines (e.g., always predict mean tip).",
            "Train/test splits and why train performance alone is not enough.",
            "Comparing RMSE, MAE, and R² across models.",
            "Writing short, stakeholder-friendly conclusions."
        ],
    },
    {
        "title": "Session 9 — Communication and Storytelling with Data",
        "goals": [
            "Turn models and plots into a coherent story.",
            "Choose appropriate charts for the question you’re answering.",
            "Practice writing ELI5 explanations for non-technical readers."
        ],
        "concepts": [
            "Framing analysis around questions, not just methods.",
            "Chart selection: when to use scatter, boxplot, bar, line, etc.",
            "Annotating charts and adding takeaways in titles/captions.",
            "Structuring a short written summary (context → findings → implications)."
        ],
    },
    {
        "title": "Session 10 — Capstone: Mini Case Study on Tipping Behavior",
        "goals": [
            "Pull together EDA, modeling, and diagnostics into one workflow.",
            "Produce a short written report or notebook telling the tipping story.",
            "Identify next questions and limitations of the current analysis."
        ],
        "concepts": [
            "End-to-end workflow: load → clean → explore → model → diagnose → communicate.",
            "Version control with Git/GitHub for notebooks and reports.",
            "Saving figures and using a simple plot index for navigation.",
            "Reflecting on what you’ve learned and where to go next."
        ],
    },
]

# ---------------------------
# 2. Build the PDF
# ---------------------------

file_name = "Study_Curriculum_Outline.pdf"
doc = SimpleDocTemplate(
    file_name,
    pagesize=letter,
    rightMargin=0.75 * inch,
    leftMargin=0.75 * inch,
    topMargin=0.75 * inch,
    bottomMargin=0.75 * inch,
)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "Title",
    parent=styles["Title"],
    fontSize=22,
    leading=26,
    spaceAfter=18,
)

session_title_style = ParagraphStyle(
    "SessionTitle",
    parent=styles["Heading2"],
    fontSize=14,
    leading=18,
    spaceBefore=12,
    spaceAfter=6,
)

section_label_style = ParagraphStyle(
    "SectionLabel",
    parent=styles["Normal"],
    fontSize=11,
    leading=14,
    spaceBefore=4,
    spaceAfter=2,
    textColor="#333333",
)

body_style = ParagraphStyle(
    "Body",
    parent=styles["Normal"],
    fontSize=10.5,
    leading=13,
)

story = []

# Cover title
title_text = "DIY Data Science Curriculum — High-Level Outline"
story.append(Paragraph(title_text, title_style))

date_text = f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M')}"
story.append(Paragraph(date_text, styles["Normal"]))
story.append(Spacer(1, 0.3 * inch))

intro = (
    "This document summarizes the high-level course structure we've been building "
    "around the restaurant tips dataset. Each session lists practical learning goals "
    "and core concepts to revisit."
)
story.append(Paragraph(intro, body_style))
story.append(PageBreak())

# ---------------------------
# Sessions
# ---------------------------

for i, session in enumerate(sessions, start=1):

    story.append(Paragraph(session["title"], session_title_style))

    # --- Learning Goals ---
    story.append(Paragraph("<b>Learning Goals</b>", section_header))

    goal_items = []
    for g in session["goals"]:
        # Unicode bullet makes it guaranteed visible
        goal_items.append(
            ListItem(Paragraph(f"{g}", bullet_style), value="•")
        )

    story.append(ListFlowable(
        goal_items,
        bulletType='bullet',
        leftIndent=12
    ))
    story.append(Spacer(1, 0.15 * inch))

    # --- Concepts ---
    story.append(Paragraph("<b>Key Concepts</b>", section_header))

    concept_items = []
    for c in session["concepts"]:
        concept_items.append(
            ListItem(Paragraph(f"{c}", bullet_style), value="•")
        )

    story.append(ListFlowable(
        concept_items,
        bulletType='bullet',
        leftIndent=12
    ))

    story.append(Spacer(1, 0.25 * inch))

    if i % 3 == 0:
        story.append(PageBreak())

# Build the PDF
doc.build(story)
print(f"✅ PDF generated: {file_name}")


✅ PDF generated: Study_Curriculum_Outline.pdf
